# PayRoute AI — Phase 2: Exploratory Data Analysis & Feature Engineering

**Objective**: Analyze the 100,000+ synthetic payment transaction dataset, investigate underlying failure drivers, perform rigorous data-leakage auditing, and build a reusable Scikit-Learn feature engineering pipeline.

---
### Table of Contents
1. **Environment Setup & Dataset Ingestion**
2. **Dataset Structure & Data Quality Checks**
3. **Target Variable Analysis (`payment_status`)**
4. **Post-Outcome Diagnostic Analysis (`failure_reason`)**
5. **Categorical & Infrastructure Bivariate Analysis**
6. **Continuous Attributes & Latency Quantiles**
7. **Data Leakage & Pre-Transaction Availability Audit**
8. **Chronological Train / Validation / Test Splitting**
9. **Feature Engineering Pipeline Demonstration**
10. **Key Insights & Next Steps**

## 1. Environment Setup & Dataset Ingestion

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project root is accessible
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.features.build_features import (
    create_feature_pipeline,
    get_feature_names,
    split_data_chronologically,
    transform_single_transaction,
)

# Set clean visualization styling
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.dpi"] = 120

# Ingest dataset from data/raw
data_path = PROJECT_ROOT / "data" / "raw" / "payments_synthetic.csv"
df = pd.read_csv(data_path)
print(f"Loaded dataset with {len(df):,} rows and {len(df.columns)} columns from {data_path.name}.")
df.head(5)

## 2. Dataset Structure & Data Quality Checks

In [ ]:
# Summary information and missing value check
print("=== Missing Values Summary ===")
print(df.isnull().sum())

print("\n=== Data Quality & Boundary Checks ===")
print(f"Total rows: {len(df):,}")
print(f"Unique transaction IDs: {df['transaction_id'].nunique():,}")
print(f"Amount range: ₹{df['amount'].min():.2f} to ₹{df['amount'].max():,.2f}")
print(f"Bank Latency range: {df['bank_latency_ms'].min()}ms to {df['bank_latency_ms'].max()}ms")
print(f"Gateway Latency range: {df['gateway_latency_ms'].min()}ms to {df['gateway_latency_ms'].max()}ms")
print(f"Negative latency instances: {(df['bank_latency_ms'] < 0).sum() + (df['gateway_latency_ms'] < 0).sum()}")

## 3. Target Variable Analysis (`payment_status`)

`payment_status`: 0 = SUCCESS, 1 = FAILED

In [ ]:
status_counts = df["payment_status"].value_counts()
status_pcts = df["payment_status"].value_counts(normalize=True) * 100.0

target_summary = pd.DataFrame({
    "Count": status_counts,
    "Percentage (%)": status_pcts.round(2)
})
target_summary.index = ["SUCCESS (0)", "FAILED (1)"]
print(target_summary)

fig, ax = plt.subplots(figsize=(6, 3.5))
colors = ["#2ca02c", "#d62728"]
bars = ax.bar(target_summary.index, target_summary["Count"], color=colors, width=0.45, edgecolor="black")
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 1200, f"{h:,}\n({h/len(df)*100:.1f}%)", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set_title("Payment Outcome Distribution (Target Variable)", fontweight="bold")
ax.set_ylabel("Number of Transactions")
ax.set_ylim(0, len(df) * 1.05)
plt.tight_layout()
plt.show()

## 4. Post-Outcome Diagnostic Analysis (`failure_reason`)

> **Anti-Leakage Note**: `failure_reason` is generated post-facto and is only populated when `payment_status == 1`. It is excluded from Stage 1 failure probability prediction.

In [ ]:
failures_df = df[df["payment_status"] == 1]
reason_counts = failures_df["failure_reason"].value_counts()
reason_pct = (reason_counts / len(failures_df)) * 100.0

pd.DataFrame({"Failure Count": reason_counts, "Share of Failures (%)": reason_pct.round(2)})

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(reason_counts.index[::-1], reason_counts.values[::-1], color="#3b528b", edgecolor="black")
for bar in bars:
    w = bar.get_width()
    ax.text(w + 50, bar.get_y() + bar.get_height()/2, f"{w:,} ({w/len(failures_df)*100:.1f}%)", ha="left", va="center", fontsize=8, fontweight="bold")
ax.set_title("Failure Reasons Breakdown (Conditional on Payment Failure)", fontweight="bold")
ax.set_xlabel("Count")
ax.set_xlim(0, max(reason_counts.values) * 1.25)
plt.tight_layout()
plt.show()

## 5. Categorical Bivariate Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# 1. Payment Method
meth = df.groupby("payment_method")["payment_status"].mean() * 100.0
axes[0, 0].bar(meth.index, meth.values, color="#1f77b4", edgecolor="black")
axes[0, 0].set_title("Failure Rate by Payment Method (%)", fontweight="bold")
axes[0, 0].set_ylabel("Failure Rate (%)")

# 2. Bank
bank = df.groupby("bank")["payment_status"].mean() * 100.0
axes[0, 1].bar(bank.index, bank.values, color="#ff7f0e", edgecolor="black")
axes[0, 1].set_title("Failure Rate by Issuing Bank (%)", fontweight="bold")

# 3. Network Type
net = df.groupby("network_type")["payment_status"].mean().reindex(["2G", "3G", "4G", "5G", "WIFI"]) * 100.0
axes[1, 0].bar(net.index, net.values, color="#d62728", edgecolor="black")
axes[1, 0].set_title("Failure Rate by Network Type (%)", fontweight="bold")
axes[1, 0].set_ylabel("Failure Rate (%)")

# 4. Device Type
dev = df.groupby("device_type")["payment_status"].mean() * 100.0
axes[1, 1].bar(dev.index, dev.values, color="#9467bd", edgecolor="black")
axes[1, 1].set_title("Failure Rate by Device Type (%)", fontweight="bold")

plt.tight_layout()
plt.show()

## 6. Continuous Attributes & Latency Analysis

In [ ]:
# Latency Quintile Analysis
df["bank_lat_quintile"] = pd.qcut(df["bank_latency_ms"], q=5)
lat_analysis = df.groupby("bank_lat_quintile", observed=False)["payment_status"].agg(["mean", "count"]).reset_index()
lat_analysis["failure_rate_pct"] = lat_analysis["mean"] * 100.0

print("=== Bank Latency Quintiles vs Observed Failure Rate ===")
print(lat_analysis[["bank_lat_quintile", "count", "failure_rate_pct"]])

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot([f"Q{i+1}" for i in range(5)], lat_analysis["failure_rate_pct"], marker="o", linewidth=2.5, color="#e6550d")
ax.set_title("Monotonic Failure Probability Growth across Bank Latency Quintiles", fontweight="bold")
ax.set_xlabel("Latency Quintile (Low to High Latency)")
ax.set_ylabel("Observed Failure Rate (%)")
plt.tight_layout()
plt.show()

## 7. Data Leakage & Pre-Transaction Availability Audit

Before training any ML model, we classify all attributes by their temporal point of observation:

1. **Pre-Transaction Request Context**: `amount`, `payment_method`, `bank`, `merchant_category`, `device_type`, `network_type`, `customer_age_days`, `previous_transactions`, `previous_failed_transactions`, `previous_attempts`, `transaction_velocity`, `is_new_device`.
2. **Pre-Transaction Infrastructure Telemetry**: `bank_success_rate` (5m rolling), `gateway_success_rate` (5m rolling), estimated route latencies.
3. **Excluded Post-Outcome Diagnostics**: `payment_status` (Target $y$), `failure_reason` (Post-outcome diagnostic only).

## 8. Chronological Train / Validation / Test Splitting

In [ ]:
train_df, val_df, test_df = split_data_chronologically(df, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15)

print(f"Train Partition: {len(train_df):,} rows ({train_df['payment_status'].mean()*100:.2f}% failure rate)")
print(f"Validation Partition: {len(val_df):,} rows ({val_df['payment_status'].mean()*100:.2f}% failure rate)")
print(f"Test Partition: {len(test_df):,} rows ({test_df['payment_status'].mean()*100:.2f}% failure rate)")

## 9. Feature Engineering Pipeline Demonstration

In [ ]:
# Build and fit pipeline on Train partition only (Zero lookahead leakage)
pipeline = create_feature_pipeline()
X_train = pipeline.fit_transform(train_df)
X_val = pipeline.transform(val_df)
X_test = pipeline.transform(test_df)
feature_names = get_feature_names(pipeline)

print(f"Pipeline fitted successfully!")
print(f"Transformed Training Matrix Shape: {X_train.shape}")
print(f"Transformed Validation Matrix Shape: {X_val.shape}")
print(f"Transformed Test Matrix Shape: {X_test.shape}")
print(f"Total Output Features: {len(feature_names)}")

# Sample Feature Names
print("\nSample Engineered Feature Names:")
for feat in feature_names[:15]:
    print(f" - {feat}")
print(" ... and more.")

In [ ]:
# Real-Time Single Transaction Inference Demonstration
sample_payment = {
    "amount": 3450.0,
    "currency": "INR",
    "payment_method": "UPI",
    "bank": "HDFC",
    "merchant_category": "ECOMMERCE",
    "hour": 14,
    "day_of_week": 3,
    "device_type": "MOBILE",
    "network_type": "5G",
    "customer_age_days": 420,
    "previous_transactions": 18,
    "previous_failed_transactions": 2,
    "previous_attempts": 0,
    "transaction_velocity": 1,
    "is_new_device": 0,
    "bank_latency_ms": 180,
    "gateway_latency_ms": 110,
    "bank_success_rate": 0.94,
    "gateway_success_rate": 0.96,
}

X_single = transform_single_transaction(sample_payment, pipeline)
print(f"Single transaction transformed vector shape: {X_single.shape}")
print(f"Vector non-zero elements: {np.count_nonzero(X_single)} of {X_single.shape[1]}")

## 10. Key Insights & Phase 2 Conclusion

1. **Clean Class Distribution**: Overall failure rate is calibrated at **14.72%** (Success: 85.28%), matching real-world fintech payment distributions.
2. **Distinct Failure Modes**: The 5 failure reasons reflect heterogeneous underlying conditions (insufficient funds, latency timeouts, authentication drops, network connection drops, bank downtimes).
3. **Zero Target Leakage**: Post-outcome diagnostics and raw targets are strictly segregated from the feature matrix.
4. **Production Feature Pipeline**: The custom `ColumnTransformer` handles cyclic time transformations, causal customer history, composite infrastructure telemetry, amount scaling, and unseen categorical values.

Ready for **Phase 3: Machine Learning Model Development & Explainability**.